# Laboratorio: vectores, geometría de $\mathbb R^n$, matrices y sistemas

Este cuaderno acompaña las clases C1 y C2. La teoría y las demostraciones se encuentran en la hoja de lectura. Aquí vamos a calcular, visualizar, formular conjeturas y comprobar propiedades.

Al finalizar deberías poder interpretar cada resultado, no solamente obtenerlo con Python.

In [ ]:
import math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## C1. Vectores y geometría en $\mathbb R^n$

Comenzamos comparando operaciones implementadas directamente con listas y las operaciones de `numpy`.

In [ ]:
def sumar_vectores(u, v):
    if len(u) != len(v):
        raise ValueError("Los vectores deben tener la misma dimensión")
    return [ui + vi for ui, vi in zip(u, v)]

def multiplicar_por_escalar(alpha, u):
    return [alpha * ui for ui in u]

u_lista = [2, -1, 3]
v_lista = [1, 4, -2]

print("Con listas:", sumar_vectores(u_lista, v_lista))
print("Con listas:", multiplicar_por_escalar(-2, u_lista))

u = np.array(u_lista, dtype=float)
v = np.array(v_lista, dtype=float)
print("Con NumPy:", u + v)
print("Con NumPy:", -2 * u)

### Visualización de suma y múltiplos

En el plano, la suma puede interpretarse mediante la regla del paralelogramo.

In [ ]:
def dibujar_vectores(vectores, etiquetas, colores=None, titulo="Vectores"):
    colores = colores or [None] * len(vectores)
    fig, ax = plt.subplots(figsize=(6, 6))
    for vector, etiqueta, color in zip(vectores, etiquetas, colores):
        ax.quiver(0, 0, vector[0], vector[1], angles="xy",
                  scale_units="xy", scale=1, color=color, label=etiqueta)
    maximo = max(1, max(np.max(np.abs(w)) for w in vectores)) + 1
    ax.set_xlim(-maximo, maximo)
    ax.set_ylim(-maximo, maximo)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_title(titulo)
    plt.show()

a = np.array([2.0, 1.0])
b = np.array([-1.0, 2.0])
def dibujar_suma_cabeza_cola(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    suma = a + b
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.quiver(0, 0, a[0], a[1], angles="xy", scale_units="xy", scale=1,
              color="tab:blue", label="a")
    ax.quiver(a[0], a[1], b[0], b[1], angles="xy", scale_units="xy", scale=1,
              color="tab:orange", label="b trasladado")
    ax.quiver(0, 0, suma[0], suma[1], angles="xy", scale_units="xy", scale=1,
              color="tab:green", label="a+b")
    ax.plot([0, b[0], suma[0]], [0, b[1], suma[1]],
            color="gray", linestyle="--", alpha=0.6)
    maximo = max(1, np.max(np.abs(np.concatenate([a, b, suma])))) + 1
    ax.set_xlim(-maximo, maximo)
    ax.set_ylim(-maximo, maximo)
    ax.axhline(0, color="black", linewidth=0.7)
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_title("Suma mediante la regla cabeza-cola")
    plt.show()

dibujar_suma_cabeza_cola(a, b)
dibujar_vectores([a, 2*a, -1.5*a], ["a", "2a", "-1.5a"],
                  ["tab:blue", "tab:green", "tab:red"],
                  "Múltiplos de un vector")

### Norma, distancia y normalización

In [ ]:
def norma_sin_numpy(x):
    return math.sqrt(sum(xi**2 for xi in x))

def normalizar(x, tolerancia=1e-12):
    x = np.asarray(x, dtype=float)
    norma = np.linalg.norm(x)
    if norma < tolerancia:
        raise ValueError("El vector cero no se puede normalizar")
    return x / norma

x = np.array([3.0, 4.0])
y = np.array([-1.0, 2.0])

print("Norma sin NumPy:", norma_sin_numpy(x))
print("Norma con NumPy:", np.linalg.norm(x))
print("Distancia entre x e y:", np.linalg.norm(x - y))
print("Vector unitario en la dirección de x:", normalizar(x))
print("Norma del vector unitario:", np.linalg.norm(normalizar(x)))

### Producto punto y Cauchy-Schwarz

**Enunciado recordatorio.** Para todo $u,v\in\mathbb R^n$,

$$|u^Tv|\leq \|u\|\,\|v\|.$$

A continuación calcularemos ambos lados en varios ejemplos. Estas comprobaciones numéricas ilustran el teorema, pero no constituyen una prueba. La prueba se encuentra en la hoja teórica.

In [ ]:
def producto_punto_sin_numpy(u, v):
    if len(u) != len(v):
        raise ValueError("Los vectores deben tener la misma dimensión")
    return sum(ui * vi for ui, vi in zip(u, v))

u_prueba = [1, 2, -1]
v_prueba = [3, 0, 2]
print("Producto punto sin NumPy:", producto_punto_sin_numpy(u_prueba, v_prueba))
print("Producto punto con NumPy:", np.array(u_prueba) @ np.array(v_prueba))

def verificar_cauchy_schwarz(u, v, tolerancia=1e-12):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    izquierda = abs(u @ v)
    derecha = np.linalg.norm(u) * np.linalg.norm(v)
    return izquierda, derecha, izquierda <= derecha + tolerancia

pares = [
    (np.array([1, 2, -1]), np.array([3, 0, 2])),
    (np.array([1, 2, 3]), np.array([2, 4, 6])),
    (np.array([1, 0]), np.array([0, 1])),
]

for u, v in pares:
    izquierda, derecha, se_cumple = verificar_cauchy_schwarz(u, v)
    print(f"u={u}, v={v}: {izquierda:.4f} <= {derecha:.4f}: {se_cumple}")

### Ángulo y ortogonalidad

El recorte numérico con `np.clip` evita que los errores de redondeo produzcan un valor ligeramente fuera del intervalo $[-1,1]$.

In [ ]:
def angulo_entre_vectores(u, v, grados=True):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    denominador = np.linalg.norm(u) * np.linalg.norm(v)
    if denominador == 0:
        raise ValueError("El ángulo no está definido para el vector cero")
    coseno = np.clip((u @ v) / denominador, -1.0, 1.0)
    angulo = np.arccos(coseno)
    return np.degrees(angulo) if grados else angulo

ejemplos = [
    (np.array([1, 0]), np.array([1, 1])),
    (np.array([1, 2]), np.array([-2, 1])),
    (np.array([1, 0]), np.array([-1, 1])),
]

for u, v in ejemplos:
    print(f"Ángulo entre {u} y {v}: {angulo_entre_vectores(u, v):.2f} grados")

### Actividades de C1

1. Modifica los vectores de los ejemplos y predice el signo de su producto punto antes de ejecutar.
2. Busca dos vectores distintos para los cuales haya igualdad en Cauchy-Schwarz. Explica por qué.
3. Construye dos vectores no nulos ortogonales en $\mathbb R^3$.
4. Comprueba numéricamente la desigualdad triangular para cinco pares aleatorios.
5. Explica por qué el ángulo con el vector cero no está definido.

## C2. Matrices y sistemas sencillos

In [ ]:
A = np.array([[1., 2., 0.],
              [0., -1., 3.]])
B = np.array([[2., 1.],
              [1., 0.],
              [-1., 4.]])

print("A tiene forma", A.shape)
print("B tiene forma", B.shape)
print("AB =\n", A @ B)
print("(AB)^T =\n", (A @ B).T)
print("B^T A^T =\n", B.T @ A.T)
print("¿Se verifica (AB)^T=B^T A^T?", np.allclose((A @ B).T, B.T @ A.T))

S = np.array([[1., 2.], [3., 4.]])
I = np.eye(2)
S_inv = np.linalg.inv(S)
print("\nS I =\n", S @ I)
print("S^{-1} =\n", S_inv)
print("S S^{-1} =\n", S @ S_inv)
print("¿La inversa cumple la identidad?", np.allclose(S @ S_inv, I))

### $Ax$ como combinación de las columnas de $A$

In [ ]:
C = np.array([[1., 0., 2.],
              [2., 1., -1.],
              [0., 3., 1.]])
x = np.array([2., -1., 3.])

producto = C @ x
combinacion = x[0]*C[:, 0] + x[1]*C[:, 1] + x[2]*C[:, 2]

print("Cx =", producto)
print("Combinación de columnas =", combinacion)
print("¿Son iguales?", np.allclose(producto, combinacion))

### Sistemas diagonales

Para una matriz diagonal con entradas no nulas, cada ecuación se resuelve de manera independiente: $x_i=b_i/a_{ii}$. Conservamos las dos perspectivas del cuaderno original: cálculo vectorizado y recorrido explícito con fracciones.

In [ ]:
def resolver_diagonal(D, b):
    D = np.asarray(D, dtype=float)
    b = np.asarray(b, dtype=float)
    if D.ndim != 2 or D.shape[0] != D.shape[1] or D.shape[0] != len(b):
        raise ValueError("Las dimensiones de D y b no son compatibles")
    if not np.allclose(D, np.diag(np.diag(D))):
        raise ValueError("D debe ser diagonal")
    diagonal = np.diag(D)
    if np.any(np.isclose(diagonal, 0)):
        raise ValueError("Este procedimiento requiere diagonal no nula")
    return b / diagonal

D_1 = np.diag([2., 5., -3.])
b_1 = np.array([4., 10., -6.])
x_1 = resolver_diagonal(D_1, b_1)
print("Primer sistema:")
print("x =", x_1)
print("residuo b-Dx =", b_1 - D_1 @ x_1)

D_2 = np.diag([3., 2., -1.])
b_2 = np.array([7., 8., 4.])
x_2 = np.zeros(len(b_2))
for i in range(len(b_2)):
    x_2[i] = b_2[i] / D_2[i, i]

print("\nSegundo sistema, expresado con fracciones:")
print([str(Fraction(valor).limit_denominator()) for valor in x_2])
print("residuo b-Dx =", b_2 - D_2 @ x_2)

### Sistemas triangulares

Implementamos sustitución hacia adelante y hacia atrás siguiendo directamente las fórmulas recursivas. Además de la solución, las funciones devuelven un conteo elemental de multiplicaciones, sumas/restas y divisiones.

In [ ]:
def sustitucion_adelante(L, b):
    L = np.asarray(L, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n)
    operaciones = 0
    for i in range(n):
        if np.isclose(L[i, i], 0):
            raise ValueError(f"Pivote diagonal nulo en la posición {i}")
        suma = 0.0
        for j in range(i):
            suma += L[i, j] * x[j]
            operaciones += 2
        x[i] = (b[i] - suma) / L[i, i]
        operaciones += 2
    return x, operaciones

def sustitucion_atras(U, b):
    U = np.asarray(U, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(b)
    x = np.zeros(n)
    operaciones = 0
    for i in range(n - 1, -1, -1):
        if np.isclose(U[i, i], 0):
            raise ValueError(f"Pivote diagonal nulo en la posición {i}")
        suma = 0.0
        for j in range(i + 1, n):
            suma += U[i, j] * x[j]
            operaciones += 2
        x[i] = (b[i] - suma) / U[i, i]
        operaciones += 2
    return x, operaciones

L = np.array([[2., 0., 0.], [1., 3., 0.], [4., 2., -1.]])
b_L = np.array([4., 5., -4.])
x_L, ops_L = sustitucion_adelante(L, b_L)

U = np.array([[2., -1., 3.], [0., 4., 2.], [0., 0., 5.]])
b_U = np.array([9., 10., 15.])
x_U, ops_U = sustitucion_atras(U, b_U)

print("Sustitución hacia adelante")
print("x =", x_L, "; operaciones =", ops_L)
print("residuo b-Lx =", b_L - L @ x_L)
print("\nSustitución hacia atrás")
print("x =", x_U, "; operaciones =", ops_U)
print("residuo b-Ux =", b_U - U @ x_U)

### Operaciones elementales de fila

Construimos la matriz aumentada y aplicamos las tres operaciones. Cada función trabaja sobre una copia para no modificar accidentalmente los datos originales. Después aplicamos la operación inversa y comprobamos que recuperamos la matriz de partida.

In [ ]:
def matriz_aumentada(A, b):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float).reshape(-1)
    if A.shape[0] != len(b):
        raise ValueError("A y b deben tener el mismo número de filas")
    return np.column_stack([A, b])

def intercambiar_filas(M, i, j):
    R = np.array(M, dtype=float, copy=True)
    R[[i, j]] = R[[j, i]]
    return R

def escalar_fila(M, i, alpha):
    if np.isclose(alpha, 0):
        raise ValueError("El escalar debe ser no nulo")
    R = np.array(M, dtype=float, copy=True)
    R[i] *= alpha
    return R

def sumar_multiplo(M, destino, origen, alpha):
    R = np.array(M, dtype=float, copy=True)
    R[destino] += alpha * R[origen]
    return R

A = np.arange(1, 21, dtype=float).reshape(4, 5)
b = np.array([1., 2., 3., 4.])
Ab = matriz_aumentada(A, b)

print("Matriz aumentada original:\n", Ab)

escalada = escalar_fila(Ab, i=0, alpha=2)
recuperada_1 = escalar_fila(escalada, i=0, alpha=1/2)
print("\nF1 <- 2 F1:\n", escalada)
print("¿La operación inversa recupera Ab?", np.allclose(recuperada_1, Ab))

sumada = sumar_multiplo(Ab, destino=2, origen=1, alpha=3)
recuperada_2 = sumar_multiplo(sumada, destino=2, origen=1, alpha=-3)
print("\nF3 <- F3 + 3 F2:\n", sumada)
print("¿La operación inversa recupera Ab?", np.allclose(recuperada_2, Ab))

intercambiada = intercambiar_filas(Ab, 0, 3)
recuperada_3 = intercambiar_filas(intercambiada, 0, 3)
print("\nF1 <-> F4:\n", intercambiada)
print("¿La operación inversa recupera Ab?", np.allclose(recuperada_3, Ab))

### Matrices elementales

Aplicar una operación elemental a una matriz $M$ equivale a multiplicarla por la izquierda por la matriz obtenida al aplicar esa operación a la identidad. Verificaremos $E(M)=E(I)M$ para las tres operaciones.

In [ ]:
m = Ab.shape[0]
I = np.eye(m)

E_escalar = escalar_fila(I, i=0, alpha=2)
E_suma = sumar_multiplo(I, destino=2, origen=1, alpha=3)
E_intercambio = intercambiar_filas(I, 0, 3)

pruebas = [
    ("F1 <- 2 F1", E_escalar, escalar_fila(Ab, 0, 2)),
    ("F3 <- F3 + 3 F2", E_suma, sumar_multiplo(Ab, 2, 1, 3)),
    ("F1 <-> F4", E_intercambio, intercambiar_filas(Ab, 0, 3)),
]

for nombre, E, resultado_directo in pruebas:
    print(f"\n{nombre}")
    print("Matriz elemental E(I):\n", E)
    print("¿E(I) Ab coincide con la operación directa?",
          np.allclose(E @ Ab, resultado_directo))
    print("¿E es invertible?", np.linalg.matrix_rank(E) == m)

### Resolución completa mediante operaciones elementales

En este ejemplo elegimos las operaciones manualmente. La próxima clase convertirá esta elección en el algoritmo sistemático de eliminación de Gauss-Jordan.

In [ ]:
A_sistema = np.array([[1., 2.], [3., 4.]])
b_sistema = np.array([5., 11.])
M0 = matriz_aumentada(A_sistema, b_sistema)

M1 = sumar_multiplo(M0, destino=1, origen=0, alpha=-3)
M2 = escalar_fila(M1, i=1, alpha=-1/2)
M3 = sumar_multiplo(M2, destino=0, origen=1, alpha=-2)

for numero, M in enumerate([M0, M1, M2, M3]):
    print(f"Paso {numero}:\n{M}\n")

x_solucion = M3[:, -1]
print("Solución leída de la matriz final:", x_solucion)
print("Residuo b-Ax:", b_sistema - A_sistema @ x_solucion)
print("Comparación con np.linalg.solve:", np.linalg.solve(A_sistema, b_sistema))

### Actividades de C2

1. Enumera cuáles propiedades del producto de números reales se conservan para matrices y cuáles fallan.
2. Construye matrices para las que $AB$ esté definido, pero $BA$ no.
3. Verifica con un ejemplo que, en general, $AB\ne BA$.
4. Modifica `resolver_diagonal` para distinguir entre una ecuación incompatible y una variable libre cuando aparece un cero en la diagonal.
5. Modifica los sistemas triangulares y comprueba siempre el residuo $b-Ax$.
6. Compara el conteo de operaciones para matrices triangulares de órdenes 3, 10 y 100.
7. Construye explícitamente la matriz elemental para $F_2\leftarrow -4F_2$ y su inversa.
8. Construye la matriz elemental para $F_1\leftrightarrow F_3$ y verifica que es su propia inversa.
9. Resuelve un sistema $3\times3$ mediante las funciones elementales y registra cada paso.
10. Explica por qué no permitimos multiplicar una fila por cero.
11. Explica la diferencia entre verificar $Ax=b$ y ejecutar simplemente `np.linalg.solve`.

La próxima clase sistematizará este procedimiento mediante eliminación de Gauss-Jordan.